### Step 1: Install dependencies

<br>

First make sure you have created a virtual environment (venv) and connected to it in this notebook (top right corner)

In [ ]:
!pip install langchain_core langchain-anthropic langgraph

### Step 2: Initialize the LLM

<br>

Create a .env with:
<br>
ANTHROPIC_API_KEY=your-api-key
<br>
LANGSMITH_TRACING=true

In [ ]:
import os
import getpass

from langchain_anthropic import ChatAnthropic

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("ANTHROPIC_API_KEY")

llm = ChatAnthropic(model="claude-haiku-4-5")

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt="""
You are a legal analyst that classifies legal documents as either in-scope or out-of-scope for processing. 
In-scope documents are SAAS agreements and Asset Purchase Agreements. 
Out-of-scope documents are anything else.
"""

user_prompt="""Here is a snippet from a legal document:

This Software as a Service Agreement (this "Agreement") is entered into as of January 10,
2024 (the "EKective Date"), by and between:
CloudPlatform Solutions Inc., a Delaware corporation ("Vendor"), with its principal place of
business at 800 Cloud Drive, Seattle, WA 98101
and
Acme Legal LLP, a limited liability partnership organized under the laws of New York
("Customer"), with its principal place of business at 425 Park Avenue, New York, NY 10022
RECITALS
WHEREAS, Vendor provides cloud-based legal practice management software and related
services;
WHEREAS, Customer desires to subscribe to and use Vendor's software platform and
services; and
WHEREAS, Vendor desires to provide such services to Customer under the terms and
conditions set forth in this Agreement;
NOW, THEREFORE, in consideration of the mutual covenants and agreements hereinafter
set forth and for other good and valuable consideration, the receipt and suKiciency of
which are hereby acknowledged, the parties agree as follows:

"""

# Invoke the LLM with a list of messages
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_prompt)
]

response = llm.invoke(messages)
print(response.content)


### Step 3: Implement Structured Output

In [ ]:
# Schema for structured output
from pydantic import BaseModel, Field
from typing import Literal

class DocumentClassification(BaseModel):
    class_: Literal["in-scope", "out-of-scope"] = Field(
        ..., description="Classification of the document as either in-scope or out-of-scope for processing."
    )
    reasoning: str = Field(
        ..., description="Explanation of why the document was classified as in-scope or out-of-scope."
    )
    key_indicators: str = Field(
        ..., description="Key indicators or phrases from the document that led to this classification."
    )
    confidence: int = Field(
        ..., description="Confidence score for the classification, ranging from 0 to 10."
    )

In [ ]:
# Augment the LLM with schema for structured output
structured_llm = llm.with_structured_output(DocumentClassification)


In [ ]:
response = structured_llm.invoke(messages)
print(f"Classification: {response.class_}")
print(f"\nReasoning: {response.reasoning}")
print(f"\nKey Indicators: {response.key_indicators}")
print(f"\nConfidence: {response.confidence}")